# Exploratory Data Analysis & Data Quality Report

## 1. Introduction

Runs the full EDA pipeline on the wound image dataset: statistical properties, data quality, key characteristics. Shows the metadata file and the visualizations the script produces.

## 2. Setup

Imports the libraries and the main pipeline function, then sets the data and output paths.

In [ ]:
import os
import sys

# KAGGLE DATASET SETUP: Add code dataset directly to path
if os.path.exists("/kaggle/input/wound-seg-code-v1"):
    sys.path.insert(0, "/kaggle/input/wound-seg-code-v1")
    print("[INFO] Code dataset added to Python path")
else:
    print("[WARNING] Code dataset not found - using local imports")

import pandas as pd
from IPython.display import Image, display

# Import pipeline components (no 'src.' prefix on Kaggle)
try:
    from pipeline.eda import EDA_CONFIG, run_eda_pipeline
    from utils.environment import detect_environment
    print("[INFO] Successfully imported pipeline modules")
except ImportError as e:
    print(f"[ERROR] Failed to import pipeline modules: {e}")
    raise

# Detect environment and set paths
env = detect_environment()
print(f"Running in {env} environment")

if env == "kaggle":
    DATA_PATH = "/kaggle/input/wound-segmentation-images/data_wound_seg/"
    SOURCE_MAP_PATH = (
        "/kaggle/input/wound-segmentation-images/data_wound_seg/correspondence_table.xlsx"
    )
    OUTPUT_CSV_PATH = "/kaggle/working/image_metadata.csv"
    OUTPUT_VIZ_DIR = "/kaggle/working/visualizations/"
else:
    # Local simulated environment
    DATA_PATH = "../simulated_kaggle_data/input/wound-segmentation-images/"
    SOURCE_MAP_PATH = "../archive/Original Data sets/Wound images segmentation [2760 samples]/correspondence_table.xlsx"
    OUTPUT_CSV_PATH = "../simulated_kaggle_data/working/image_metadata.csv"
    OUTPUT_VIZ_DIR = "../simulated_kaggle_data/working/visualizations/"

In [ ]:
# VALIDATION: Verify all imports succeeded
import sys

print("=" * 70)
print("IMPORT VALIDATION")
print("=" * 70)

required_modules = {
    'pipeline.eda': ['EDA_CONFIG', 'run_eda_pipeline'],
    'utils.environment': ['detect_environment'],
    'pandas': ['DataFrame'],
}

all_valid = True
for module_name, attributes in required_modules.items():
    try:
        module = sys.modules[module_name]
        for attr in attributes:
            if not hasattr(module, attr):
                print(f"FAIL {module_name}.{attr} - NOT FOUND")
                all_valid = False
            else:
                print(f"OK {module_name}.{attr} - OK")
    except KeyError:
        print(f"FAIL {module_name} - MODULE NOT IMPORTED")
        all_valid = False

print("=" * 70)
if all_valid:
    print("OK All imports validated successfully")
else:
    print("FAIL Import validation FAILED - stopping execution")
    raise ImportError("Required modules not available")
print("=" * 70)

## 3. Execute EDA Pipeline

One function call runs the analysis, writes the metadata CSV and creates the visualization files.

In [ ]:
pipeline_result = run_eda_pipeline(
    data_path=DATA_PATH,
    source_map_path=SOURCE_MAP_PATH,
    output_csv_path=OUTPUT_CSV_PATH,
    output_viz_dir=OUTPUT_VIZ_DIR,
    config=EDA_CONFIG,
)

# Report final status
if pipeline_result["success"]:
    print("[SUCCESS] Pipeline completed successfully.")
else:
    print(f"[ERROR] Pipeline failed: {pipeline_result['error']}")

## 4. Load and Display Metadata

The pipeline wrote `image_metadata.csv`. Loaded into a pandas DataFrame below.

In [ ]:
if os.path.exists(OUTPUT_CSV_PATH):
    df = pd.read_csv(OUTPUT_CSV_PATH)
    display(df.head())
else:
    print("Metadata CSV file not found.")

In [ ]:
if "df" in locals():
    display(df.describe())

## 5. Display Summary Table

The pipeline also wrote a summary table of key dataset statistics.

In [ ]:
# Display the summary table from the pipeline result
if pipeline_result["success"] and pipeline_result["data"]["summary_table_result"]:
    summary_result = pipeline_result["data"]["summary_table_result"]
    if summary_result["success"]:
        print("[INFO] Dataset Summary Statistics")
        print("=" * 50)
        summary_df = summary_result["data"]
        display(summary_df)
    else:
        print(f"[ERROR] Summary table generation failed: {summary_result['error']}")
else:
    print("[WARNING] No summary table available")

## 6. Display Visualizations

The pipeline also wrote plots to the output directory. Displayed below.

In [ ]:
# Dynamically find and display all PNG visualizations
import glob

viz_files = sorted(glob.glob(os.path.join(OUTPUT_VIZ_DIR, "*.png")))

if viz_files:
    print(f"[INFO] Found {len(viz_files)} visualization files\n")
    for viz_path in viz_files:
        file_name = os.path.basename(viz_path)
        print(f"--- {file_name} ---")
        display(Image(filename=viz_path))
        print()  # Add spacing between images
else:
    print("[WARNING] No visualization files found in output directory")

## 7. Conclusion

This section will be filled in after a manual review of the data and plots above.